# OR Intelligence · Instrument detection model (all-in-one Colab notebook)

**Everything for the model happens in this notebook.** You do not need the `src/` scripts or a terminal.

| Step | What happens | WBS |
|---|---|---|
| 1 | Settings (the only cell you edit) | — |
| 2 | GPU check, install, connect Google Drive | — |
| 3 | Unzip the dataset from Drive | 2.1.1 |
| 4 | Convert Pascal-VOC boxes → YOLO labels (6 instruments) | 2.2.1 |
| 5 | Split **by video** into train / val / test | 2.2.2 |
| 6 | Look at the data (counts + example images) | 2.4.1 |
| 7 | Train the baseline model | 3.1.1 |
| 8 | Optional: one experiment | 3.1.2 |
| 9 | Pick the best model (validation only) | 3.1.2 |
| 10 | Test-set evaluation (**run once**) | 3.2.1 |
| 11 | Save the model to Drive (+ download) | 3.3.2 |
| 12 | Run the model on a video → `detections.jsonl` for the app | 3.3.1 |

**How to use**
1. `Runtime → Change runtime type → T4 GPU`.
2. Upload the dataset zip (m2cai16-tool-locations) to your Google Drive once, and set `DATA_ZIP` in step 1.
3. `Runtime → Run all`, or run the cells top to bottom.

Everything important is written to Google Drive (`DRIVE_DIR`), so a disconnect does not lose the work. Re-running is safe.

> Educational tool, not diagnostic. Never commit the dataset or `.pt` weights to GitHub; they live in Drive.

## 1 · Settings — the only cell you need to edit

In [ ]:
# ---- where things live in your Google Drive -------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/or-intelligence"          # everything is saved under here
DATA_ZIP  = f"{DRIVE_DIR}/data/m2cai16-tool-locations.zip"    # upload the dataset zip here once

# ---- training -----------------------------------------------------------------------------
MODEL    = "yolov8s.pt"   # COCO-pretrained start point (yolov8n.pt = faster, yolov8m.pt = stronger)
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16
PATIENCE = 10             # stop early if validation stops improving
SEED     = 42

# ---- split (by video, never by frame) ------------------------------------------------------
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}

# ---- the 6 instruments (must match contracts/classes.json in the repo) ---------------------
CLASSES = ["grasper", "bipolar", "hook", "scissors", "clipper", "irrigator"]
DROP    = {"specimenbag"}  # m2cai16 has a 7th class that we do not use

## 2 · GPU, install, Google Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
!pip -q install ultralytics

from google.colab import drive
drive.mount("/content/drive")

import os, json, random, re, shutil, zipfile, glob, hashlib, datetime
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

for sub in ["data", "runs", "weights", "results"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
WORK = Path("/content/work")      # fast local disk for images during training
WORK.mkdir(exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

## 3 · Unzip the dataset (WBS 2.1.1)

In [ ]:
RAW = WORK / "raw"
if not RAW.exists():
    assert os.path.exists(DATA_ZIP), f"Upload the dataset zip to {DATA_ZIP} first"
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(RAW)

# find the VOC folders wherever they are inside the zip
xml_files = sorted(glob.glob(f"{RAW}/**/*.xml", recursive=True))
img_files = sorted(glob.glob(f"{RAW}/**/*.jpg", recursive=True) + glob.glob(f"{RAW}/**/*.png", recursive=True))
images_by_stem = {Path(p).stem: p for p in img_files}
print(f"XML annotations: {len(xml_files)}   images: {len(img_files)}")
print("example names:", [Path(p).stem for p in xml_files[:5]])

## 4 · Convert VOC boxes → YOLO labels (WBS 2.2.1)

Each box becomes `class cx cy w h` (0–1). SpecimenBag is dropped, boxes are clipped to the image, images without tools keep an empty label file.

In [ ]:
NAME_TO_ID = {n: i for i, n in enumerate(CLASSES)}
stats = Counter()

def convert(xml_path):
    root = ET.parse(xml_path).getroot()
    W = float(root.find("size/width").text); H = float(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        if name in DROP:
            stats["dropped_specimenbag"] += 1; continue
        if name not in NAME_TO_ID:
            stats[f"unknown:{name}"] += 1; continue
        b = obj.find("bndbox")
        x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, x2 = max(0, min(x1, W)), max(0, min(x2, W))
        y1, y2 = max(0, min(y1, H)), max(0, min(y2, H))
        if x2 <= x1 or y2 <= y1:
            stats["degenerate_box"] += 1; continue
        cx, cy, w, h = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H
        lines.append(f"{NAME_TO_ID[name]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        stats[CLASSES[NAME_TO_ID[name]]] += 1
    return lines

labels = {}                       # stem -> list of label lines
for x in xml_files:
    stem = Path(x).stem
    if stem not in images_by_stem:
        stats["xml_without_image"] += 1; continue
    try:
        labels[stem] = convert(x)
    except Exception as e:
        stats["malformed_xml"] += 1
print(f"converted {len(labels)} images")
print(dict(stats))

## 5 · Split by video (WBS 2.2.2)

Frames from the same surgery never end up in two splits, so the test score measures unseen surgeries. **Check the printed video IDs look right** (adjust `video_id()` if the file names use another pattern).

In [ ]:
def video_id(stem):
    # m2cai16 frames are named like <video>_<frame>; take everything before the last "_"
    m = re.match(r"(.+)_\d+$", stem)
    return m.group(1) if m else stem.split("_")[0]

videos = defaultdict(list)
for stem in labels:
    videos[video_id(stem)].append(stem)
print(f"{len(videos)} videos:", {v: len(s) for v, s in sorted(videos.items())})

def class_counts(stems):
    c = Counter()
    for s in stems:
        for line in labels[s]:
            c[CLASSES[int(line.split()[0])]] += 1
    return c

# try several seeded shuffles, keep the first where every split has every class
rng = random.Random(SEED)
vids = sorted(videos)
best = None
for attempt in range(500):
    rng.shuffle(vids)
    n = len(vids); n_tr = round(n * SPLIT["train"]); n_va = max(1, round(n * SPLIT["val"]))
    split = {"train": vids[:n_tr], "val": vids[n_tr:n_tr + n_va], "test": vids[n_tr + n_va:]}
    ok = all(all(class_counts([s for v in split[k] for s in videos[v]])[c] > 0 for c in CLASSES) for k in split)
    if ok:
        best = {k: sorted(v) for k, v in split.items()}; break
assert best, "Could not find a split with all classes in every split: check video_id() or use fewer splits"

# leakage check: no video in two splits
all_v = best["train"] + best["val"] + best["test"]
assert len(all_v) == len(set(all_v)), "a video is in more than one split!"

# write YOLO folders + dataset.yaml
DS = WORK / "yolo"
if DS.exists(): shutil.rmtree(DS)
for k, vlist in best.items():
    (DS / "images" / k).mkdir(parents=True); (DS / "labels" / k).mkdir(parents=True)
    for v in vlist:
        for s in videos[v]:
            src = images_by_stem[s]
            shutil.copy(src, DS / "images" / k / Path(src).name)
            (DS / "labels" / k / f"{s}.txt").write_text("\n".join(labels[s]))
yaml_text = f"path: {DS}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n" + \
            "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES))
(DS / "dataset.yaml").write_text(yaml_text)

# keep the split in Drive (this file can also be committed to the repo as data/splits/splits.json)
json.dump(best, open(f"{DRIVE_DIR}/data/splits.json", "w"), indent=2)
print({k: len(v) for k, v in best.items()}, "videos per split  ->  saved splits.json to Drive")

## 6 · Look at the data (WBS 2.4.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.DataFrame({k: class_counts([s for v in best[k] for s in videos[v]]) for k in best}).reindex(CLASSES).fillna(0).astype(int)
table["total"] = table.sum(axis=1)
display(table)
table.to_csv(f"{DRIVE_DIR}/results/boxes_per_class_per_split.csv")
table.drop(columns="total").plot.bar(figsize=(8, 3), title="Boxes per class per split"); plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/results/boxes_per_class_per_split.png"); plt.show()

# 4 random training images with their boxes
from PIL import Image, ImageDraw
COLORS = ["#2563eb", "#7c3aed", "#dc2626", "#16a34a", "#ea580c", "#0891b2"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_path in zip(axes, random.Random(SEED).sample(sorted((DS / "images/train").iterdir()), 4)):
    im = Image.open(img_path).convert("RGB"); d = ImageDraw.Draw(im); W, H = im.size
    for line in (DS / "labels/train" / f"{img_path.stem}.txt").read_text().splitlines():
        c, cx, cy, w, h = line.split(); c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
        d.rectangle([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], outline=COLORS[c], width=3)
        d.text(((cx - w/2) * W + 3, (cy - h/2) * H + 3), CLASSES[c], fill=COLORS[c])
    ax.imshow(im); ax.set_title(img_path.stem, fontsize=8); ax.axis("off")
plt.show()

## 7 · Train the baseline (WBS 3.1.1)

Runs are saved straight to Drive, with a checkpoint every epoch. If Colab disconnects, re-run steps 1–5 and then the **resume** cell below.

In [ ]:
from ultralytics import YOLO

def train(run_name, **overrides):
    start_from = overrides.pop("model", MODEL)
    args = dict(data=str(DS / "dataset.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
                seed=SEED, deterministic=True, project=f"{DRIVE_DIR}/runs", name=run_name, exist_ok=True,
                save_period=1)
    args.update(overrides)
    YOLO(start_from).train(**args)
    return f"{DRIVE_DIR}/runs/{run_name}/weights/best.pt"

baseline_weights = train("baseline")
print("baseline weights:", baseline_weights)

In [ ]:
# RESUME after a disconnect (only if training stopped early). Uncomment and run:
# YOLO(f"{DRIVE_DIR}/runs/baseline/weights/last.pt").train(resume=True)

## 8 · Optional: one experiment (WBS 3.1.2)

Change **one** thing, based on step 6 (for example a larger image size if tools are small). Skip this cell if you are short on time.

In [ ]:
RUN_EXPERIMENT = False      # set True to run it
if RUN_EXPERIMENT:
    exp_weights = train("exp01_imgsz832", imgsz=832)

## 9 · Pick the best model — validation split only (WBS 3.1.2)

The test split is **not** used for choosing.

In [ ]:
def val_scores(weights):
    m = YOLO(weights).val(data=str(DS / "dataset.yaml"), split="val", imgsz=IMGSZ, verbose=False)
    return {"map50": float(m.box.map50), "map50_95": float(m.box.map),
            "per_class_ap50": {CLASSES[i]: float(m.box.ap50[j]) for j, i in enumerate(m.box.ap_class_index)}}

candidates = {"baseline": f"{DRIVE_DIR}/runs/baseline/weights/best.pt"}
if os.path.exists(f"{DRIVE_DIR}/runs/exp01_imgsz832/weights/best.pt"):
    candidates["exp01_imgsz832"] = f"{DRIVE_DIR}/runs/exp01_imgsz832/weights/best.pt"

scores = {name: val_scores(w) for name, w in candidates.items()}
display(pd.DataFrame({n: {"val mAP@0.5": s["map50"], "val mAP@0.5:0.95": s["map50_95"]} for n, s in scores.items()}).T)
chosen = max(scores, key=lambda n: scores[n]["map50"])
BEST = candidates[chosen]
json.dump({"chosen": chosen, "scores": scores}, open(f"{DRIVE_DIR}/results/model_selection.json", "w"), indent=2)
print("chosen model:", chosen)

## 10 · Test-set evaluation — run ONCE (WBS 3.2.1)

Reports mAP@0.5 and precision/recall per instrument on surgeries the model never saw. The results are saved to Drive; copy them into `docs/model/EVALUATION.md` and `docs/model/test_metrics.json` in the repo.

In [ ]:
TEST_FILE = f"{DRIVE_DIR}/results/test_metrics.json"
CONF = 0.25
if os.path.exists(TEST_FILE):
    print("Test split already evaluated once - showing the saved result instead of re-running.")
    test = json.load(open(TEST_FILE))
else:
    m = YOLO(BEST).val(data=str(DS / "dataset.yaml"), split="test", imgsz=IMGSZ, conf=CONF, plots=True,
                       project=f"{DRIVE_DIR}/runs", name="test_eval", exist_ok=True)
    per_class = {}
    for j, i in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(j)
        per_class[CLASSES[i]] = {"precision": float(p), "recall": float(r), "ap50": float(ap50)}
    test = {"model": chosen, "weights_sha256": hashlib.sha256(open(BEST, "rb").read()).hexdigest(),
            "conf": CONF, "map50": float(m.box.map50), "map50_95": float(m.box.map), "per_class": per_class,
            "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds")}
    json.dump(test, open(TEST_FILE, "w"), indent=2)
display(pd.DataFrame(test["per_class"]).T.round(3))
print(f"TEST mAP@0.5 = {test['map50']:.3f}   mAP@0.5:0.95 = {test['map50_95']:.3f}")

## 11 · Save the final model (WBS 3.3.2)

`best.pt` (+ `best.onnx`) go to `DRIVE_DIR/weights/`. Share that Drive folder with the team, or attach the file to a GitHub Release `model-v1`. **Do not commit it to the repo.**

In [ ]:
final_pt = f"{DRIVE_DIR}/weights/best.pt"
shutil.copy(BEST, final_pt)
onnx_path = YOLO(final_pt).export(format="onnx", imgsz=IMGSZ)
shutil.copy(onnx_path, f"{DRIVE_DIR}/weights/best.onnx")
sha = hashlib.sha256(open(final_pt, "rb").read()).hexdigest()
open(f"{DRIVE_DIR}/weights/best.pt.sha256", "w").write(sha)
print("saved:", final_pt, "\nsha256:", sha)

# optional: download to this computer too
from google.colab import files
# files.download(final_pt)

## 12 · Run the model on a video → `detections.jsonl` (WBS 3.3.1)

Produces the exact format the timeline/API code expects (contract C4 in `contracts/detections.schema.json`). Upload a video to Drive and set `VIDEO`.

In [ ]:
import cv2

def detect_video(video_path, out_path, sample_fps=5.0, conf=0.25, weights=final_pt):
    model = YOLO(weights)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, round(fps / sample_fps))
    vid = Path(video_path).stem
    written, idx = 0, 0
    with open(out_path, "w") as f:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if idx % step == 0:
                r = model.predict(frame, conf=conf, imgsz=IMGSZ, verbose=False)[0]
                dets = [{"cls": int(c), "name": CLASSES[int(c)], "conf": round(float(s), 4),
                         "box": [round(float(v), 4) for v in b]}
                        for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxyn)]
                f.write(json.dumps({"video_id": vid, "frame": idx, "t": round(idx / fps, 3), "w": W, "h": H, "dets": dets}) + "\n")
                written += 1
            idx += 1
    cap.release()
    return {"frames": written, "seconds": total / fps}

VIDEO = f"{DRIVE_DIR}/data/sample_video.mp4"     # upload any test video here
if os.path.exists(VIDEO):
    out = f"{DRIVE_DIR}/results/{Path(VIDEO).stem}_detections.jsonl"
    print(detect_video(VIDEO, out), "->", out)
else:
    print("No video at", VIDEO, "- skip this step or upload one.")

## Done — what to hand to the team

| File in Drive | Who needs it | Where it goes in the repo |
|---|---|---|
| `weights/best.pt` | Member C (API), Member D (demo) | shared Drive / GitHub Release — **never committed** |
| `results/test_metrics.json` | Report, slides | `docs/model/test_metrics.json` |
| `results/model_selection.json`, `boxes_per_class_per_split.*` | Report, slides | `docs/model/`, `docs/data/` |
| `data/splits.json` | Report | `data/splits/splits.json` |
| `results/*_detections.jsonl` | Member C (timeline) | input to the timeline builder |

Write the numbers into `docs/model/EVALUATION.md` and `docs/model/experiments.md`; never type numbers that did not come from this notebook.